In [1]:
!pip install transformers datasets torchvision pillow tqdm scikit-learn

In [2]:
!git clone https://github.com/MKLab-ITI/image-verification-corpus.git

Cloning into 'image-verification-corpus'...
remote: Enumerating objects: 770, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 770 (delta 7), reused 16 (delta 7), pack-reused 754 (from 1)
Receiving objects: 100% (770/770), 1.00 GiB | 51.93 MiB/s, done.
Resolving deltas: 100% (146/146), done.
Updating files: 100% (71/71), done.


In [5]:
import pandas as pd
import os

data_path = "/content/image-verification-corpus/mediaeval2015/devset"

tweets_file = os.path.join(data_path, "tweets.txt")

# đọc file txt (tab-separated)
tweets_df = pd.read_csv(tweets_file, sep='\t')

tweets_df.head()

,tweetId,tweetText,userId,imageId(s),username,timestamp,label
0,263046056240115712,¿Se acuerdan de la película: “El día después d...,21226711,sandyA_fake_46,iAnnieM,Mon Oct 29 22:34:01 +0000 2012,fake
1,262995061304852481,@milenagimon: Miren a Sandy en NY! Tremenda i...,192378571,sandyA_fake_09,CarlosVerareal,Mon Oct 29 19:11:23 +0000 2012,fake
2,262979898002534400,"Buena la foto del Huracán Sandy, me recuerda a...",132303095,sandyA_fake_09,LucasPalape,Mon Oct 29 18:11:08 +0000 2012,fake
3,262996108400271360,Scary shit #hurricane #NY http://t.co/e4JLBUfH,241995902,sandyA_fake_29,Haaaaarryyy,Mon Oct 29 19:15:33 +0000 2012,fake
4,263018881839411200,My fave place in the world #nyc #hurricane #sa...,250315890,sandyA_fake_15,princess__natt,Mon Oct 29 20:46:02 +0000 2012,fake


In [7]:
images_file = os.path.join(data_path, "images.txt")

images_df = pd.read_csv(
    images_file,
    sep='\t',
    engine='python',
    on_bad_lines='skip'
)

images_df.head()

images_df.head()

,image_id,image_url,annotation,event
0,boston_fake_01,http://i.imgur.com/LvoCC5T.jpg,fake,boston
1,boston_fake_02,http://instagram.com/p/YN7_ThPXrU/,fake,boston
2,boston_fake_03,https://o.twimg.com/2/proxy.jpg?t=HBgeaHR0cDov...,fake,boston
3,boston_fake_04,http://media.tumblr.com/a813460e72a178d8127b50...,fake,boston
4,boston_fake_05,http://i.imgur.com/uxAh4wwh.jpg,fake,boston


In [8]:
tweet_feat = pd.read_csv(os.path.join(data_path, "DatasetFeatures/tweet_features.csv"))
user_feat = pd.read_csv(os.path.join(data_path, "DatasetFeatures/user_features.csv"))

In [9]:
# rename column cho dễ dùng (tùy dataset thực tế)
tweets_df.columns = [c.strip() for c in tweets_df.columns]
images_df.columns = [c.strip() for c in images_df.columns]

# xem columns
print(tweets_df.columns)
print(images_df.columns)

Index(['tweetId', 'tweetText', 'userId', 'imageId(s)', 'username', 'timestamp',
       'label'],
      dtype='object')
Index(['image_id', 'image_url', 'annotation', 'event'], dtype='object')


In [10]:
# lấy image đầu tiên nếu có nhiều ảnh
tweets_df['image_id'] = tweets_df['imageId(s)'].astype(str).apply(lambda x: x.split(",")[0])

In [11]:
df = tweets_df.merge(images_df, on='image_id', how='left')
df = df.dropna(subset=['tweetText'])

df.head()

,tweetId,tweetText,userId,imageId(s),username,timestamp,label,image_id,image_url,annotation,event
0,263046056240115712,¿Se acuerdan de la película: “El día después d...,21226711,sandyA_fake_46,iAnnieM,Mon Oct 29 22:34:01 +0000 2012,fake,sandyA_fake_46,http://distilleryimage11.ak.instagram.com/11d0...,fake,sandy
1,262995061304852481,@milenagimon: Miren a Sandy en NY! Tremenda i...,192378571,sandyA_fake_09,CarlosVerareal,Mon Oct 29 19:11:23 +0000 2012,fake,sandyA_fake_09,https://pbs.twimg.com/media/A6Y9e5sCAAAYJqS.jpg,fake,sandy
2,262979898002534400,"Buena la foto del Huracán Sandy, me recuerda a...",132303095,sandyA_fake_09,LucasPalape,Mon Oct 29 18:11:08 +0000 2012,fake,sandyA_fake_09,https://pbs.twimg.com/media/A6Y9e5sCAAAYJqS.jpg,fake,sandy
3,262996108400271360,Scary shit #hurricane #NY http://t.co/e4JLBUfH,241995902,sandyA_fake_29,Haaaaarryyy,Mon Oct 29 19:15:33 +0000 2012,fake,sandyA_fake_29,https://igcdn-photos-g-a.akamaihd.net/hphotos-...,fake,sandy
4,263018881839411200,My fave place in the world #nyc #hurricane #sa...,250315890,sandyA_fake_15,princess__natt,Mon Oct 29 20:46:02 +0000 2012,fake,sandyA_fake_15,http://distilleryimage1.ak.instagram.com/7b8cf...,fake,sandy


In [12]:
df['label'] = df['label'].astype(str).str.lower()

df['label'] = df['label'].map({
    'fake': 1,
    'real': 0
})

df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

In [14]:
from PIL import Image
from torchvision import transforms
import torch

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

def load_image(img_id):
    try:
        path = f"/content/image-verification-corpus/images/{img_id}.jpg"
        image = Image.open(path).convert("RGB")
        return transform(image)
    except:
        # fallback nếu lỗi
        return torch.zeros(3, 224, 224)

In [15]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def encode_text(text):
    return tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [16]:
import torch
from torch.utils.data import Dataset

class FakeNewsDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        text = encode_text(str(row['tweetText']))
        image = load_image(row['image_id'])

        label = torch.tensor(row['label'], dtype=torch.long)

        return {
            "input_ids": text['input_ids'].squeeze(),
            "attention_mask": text['attention_mask'].squeeze(),
            "image": image,
            "label": label
        }

In [17]:
from torch.utils.data import DataLoader

dataset = FakeNewsDataset(df)

loader = DataLoader(
    dataset,
    batch_size=8,   # quan trọng
    shuffle=True,
    num_workers=2
)

In [18]:
import torch.nn as nn
from transformers import DistilBertModel
import torchvision.models as models

class MultimodalModel(nn.Module):
    def __init__(self):
        super().__init__()

        # TEXT
        self.text_model = DistilBertModel.from_pretrained("distilbert-base-uncased")

        # IMAGE
        self.image_model = models.resnet18(pretrained=True)
        self.image_model.fc = nn.Identity()

        # FUSION
        self.fc = nn.Sequential(
            nn.Linear(768 + 512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )

    def forward(self, input_ids, attention_mask, image):
        text_feat = self.text_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state[:, 0, :]

        image_feat = self.image_model(image)

        combined = torch.cat((text_feat, image_feat), dim=1)

        return self.fc(combined)

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MultimodalModel().to(device)

# FREEZE backbone
for param in model.text_model.parameters():
    param.requires_grad = False

for param in model.image_model.parameters():
    param.requires_grad = False

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You ca

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 200MB/s]


In [22]:
from torch.cuda.amp import autocast, GradScaler

optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)
criterion = nn.CrossEntropyLoss()

scaler = GradScaler()

for epoch in range(10):
    model.train()
    total_loss = 0

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        image = batch['image'].to(device)
        label = batch['label'].to(device)

        optimizer.zero_grad()

        with autocast():
            outputs = model(input_ids, attention_mask, image)
            loss = criterion(outputs, label)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    print(f"Epoch {epoch} Loss: {total_loss/len(loader)}")

/tmp/ipykernel_880/4289788447.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_880/4289788447.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 0 Loss: 0.5045075877703757
Epoch 1 Loss: 0.4979110666753824
Epoch 2 Loss: 0.4944028489097483
Epoch 3 Loss: 0.4915682093964683
Epoch 4 Loss: 0.4881278791954161
Epoch 5 Loss: 0.4842746252569642
Epoch 6 Loss: 0.47632342829421387
Epoch 7 Loss: 0.47379881760874565
Epoch 8 Loss: 0.46967152063575135
Epoch 9 Loss: 0.4714061377674791


In [23]:
from sklearn.metrics import accuracy_score

model.eval()
preds, labels = [], []

with torch.no_grad():
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        image = batch['image'].to(device)

        outputs = model(input_ids, attention_mask, image)
        pred = outputs.argmax(dim=1).cpu().numpy()

        preds.extend(pred)
        labels.extend(batch['label'].numpy())

print("Accuracy:", accuracy_score(labels, preds))

Accuracy: 0.7978221726828432


In [30]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

print("Train:", len(train_df))
print("Test:", len(test_df))

Train: 9330
Test: 2333


In [31]:
test_dataset = FakeNewsDataset(test_df)

from torch.utils.data import DataLoader

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False
)

In [32]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        image = batch['image'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids, attention_mask, image)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

In [33]:
acc = accuracy_score(all_labels, all_preds)

precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels,
    all_preds,
    average='binary'
)

print("Accuracy:", acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

Accuracy: 0.8079725675096442
Precision: 0.8468052347959969
Recall: 0.815418828762046
F1: 0.8308157099697885


In [34]:
label_map = {0: "REAL", 1: "FAKE"}

for i in range(5):
    row = test_df.iloc[i]

    text = row['tweetText']
    true_label = row['label']

    # dùng image_id (local)
    image = load_image(row['image_id']).unsqueeze(0).to(device)

    encoded = encode_text(text)

    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask, image)
        pred = torch.argmax(outputs, dim=1).item()

    print("TEXT:", text[:80])
    print("TRUE:", label_map[true_label])
    print("PRED:", label_map[pred])
    print("-" * 50)

TEXT: Buenos y lluviosos dias..increíble!!..imagen d un tiburón q por culpa d Sandy ac
TRUE: FAKE
PRED: FAKE
--------------------------------------------------
TEXT: “@Alyssa_Milano: Thank goodness for people who are kind  #sandy http://t.co/uOk5
TRUE: REAL
PRED: REAL
--------------------------------------------------
TEXT: LES during the storm, pic that's going around. Insane. #hurricane #hurricanesand
TRUE: REAL
PRED: REAL
--------------------------------------------------
TEXT: Indeed! RT: @thecoolhunter\nAmazing cover from New York Magazine. A view of Manh
TRUE: REAL
PRED: REAL
--------------------------------------------------
TEXT: Акулы на улицах после Сэнди #sandy http://t.co/gvC6doki
TRUE: FAKE
PRED: FAKE
--------------------------------------------------


In [35]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_preds)
print(cm)

[[ 785  199]
 [ 249 1100]]
